# Практикум 3. Морфологический анализ и лемматизация

В лаборатории 3 вы сравнивали стеммер и лемматизатор, смотрели на неоднозначные разборы и проводили слово через конечный автомат. Здесь то же самое в коде:

1. напишем стеммер Snowball для русского с нуля и сверим его с `nltk`;
2. разберём неоднозначные слова и построим парадигму с помощью `pymorphy3`;
3. сравним стемминг и лемматизацию на тексте: где стеммер режет слишком много, а где слишком мало;
4. соберём маленький конечный автомат (основа → суффикс → окончание) для существительных и проверим его по `pymorphy3`.

*Практикум к лаборатории 3 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Стеммер Snowball с нуля

**Стемминг** отрезает от слова окончания и суффиксы по правилам, без словаря. Результат (стем) не обязан быть словом: «книгами» → «книг». Русский Snowball (М. Портер) работает так:

- **RV** — часть слова после первой гласной, **R2** — более дальняя зона (после второго сочетания «гласная + согласная»). Резать можно только внутри этих зон, чтобы не съесть короткий корень.
- Шаг 1: деепричастие (-ив, -ывшись…); если нет, то возвратное -ся/-сь, а затем окончание прилагательного (с возможным суффиксом причастия), глагола или существительного. Группа 1 (-в, -вши, -ла, -ть…) снимается только после «а» или «я».
- Шаг 2: конечная «и». Шаг 3: -ость/-ост в R2. Шаг 4: превосходная степень -ейш(е), «нн» → «н» или конечный мягкий знак.

Из каждого списка берётся самый длинный подходящий суффикс. Функция печатает, что снято на каждом шаге. Названия шагов — это названия списков суффиксов, а не настоящая часть речи: стеммер не знает, что «сталью» — существительное, и снимает -ю по списку глагольных окончаний.

In [ ]:
VOWELS = "аеиоуыэюя"
PERF_GERUND_1 = ["в", "вши", "вшись"]                       # только после а/я
PERF_GERUND_2 = ["ив", "ивши", "ившись", "ыв", "ывши", "ывшись"]
ADJECTIVE = ["ее", "ие", "ые", "ое", "ими", "ыми", "ей", "ий", "ый", "ой", "ем", "им", "ым", "ом",
             "его", "ого", "ему", "ому", "их", "ых", "ую", "юю", "ая", "яя", "ою", "ею"]
PARTICIPLE_1 = ["ем", "нн", "вш", "ющ", "щ"]                   # только после а/я
PARTICIPLE_2 = ["ивш", "ывш", "ующ"]
REFLEXIVE = ["ся", "сь"]
VERB_1 = ["ла", "на", "ете", "йте", "ли", "й", "л", "ем", "н", "ло", "но", "ет", "ют", "ны", "ть", "ешь", "нно"]
VERB_2 = ["ила", "ыла", "ена", "ейте", "уйте", "ите", "или", "ыли", "ей", "уй", "ил", "ыл", "им", "ым", "ен",
          "ило", "ыло", "ено", "ят", "ует", "уют", "ит", "ыт", "ены", "ить", "ыть", "ишь", "ую", "ю"]
NOUN = ["а", "ев", "ов", "ие", "ье", "е", "иями", "ями", "ами", "еи", "ии", "и", "ией", "ей", "ой", "ий", "й",
        "иям", "ям", "ием", "ем", "ам", "ом", "о", "у", "ах", "иях", "ях", "ы", "ь", "ию", "ью", "ю", "ия", "ья", "я"]
DERIVATIONAL = ["ост", "ость"]

def regions(w):
    # RV: после первой гласной; R1: после первой пары «гласная + согласная»; R2: то же внутри R1
    rv = r1 = r2 = len(w)
    for i, ch in enumerate(w):
        if ch in VOWELS:
            rv = i + 1
            break
    for i in range(1, len(w)):
        if w[i] not in VOWELS and w[i - 1] in VOWELS:
            r1 = i + 1
            break
    for i in range(r1 + 1, len(w)):
        if w[i] not in VOWELS and w[i - 1] in VOWELS:
            r2 = i + 1
            break
    return rv, r2

def longest(w, start, suffixes):
    # самый длинный суффикс из списка, целиком лежащий в зоне [start:]
    best = ""
    for s in suffixes:
        if w.endswith(s) and len(w) - len(s) >= start and len(s) > len(best):
            best = s
    return best

def strip(w, start, group1, group2=()):
    # group1 снимается только после «а»/«я», group2 — всегда
    s = longest(w, start, list(group1) + list(group2))
    if s and s not in group2:
        i = len(w) - len(s) - 1
        if not (i >= start and w[i] in "ая"):
            s = ""
    return (w[: -len(s)], s) if s else (w, "")

def snowball_ru(word, trace=False):
    w = word.lower().replace("ё", "е")
    rv, r2 = regions(w)
    log = []
    w2, s = strip(w, rv, PERF_GERUND_1, PERF_GERUND_2)
    if s:
        w = w2; log.append(("1 деепричастие", s))
    else:
        w, s = strip(w, rv, [], REFLEXIVE)
        if s: log.append(("1 возвратность", s))
        w2, s = strip(w, rv, [], ADJECTIVE)
        if s:
            w = w2; log.append(("1 прилагательное", s))
            w, s = strip(w, rv, PARTICIPLE_1, PARTICIPLE_2)
            if s: log.append(("1 причастие", s))
        else:
            w2, s = strip(w, rv, VERB_1, VERB_2)
            if s:
                w = w2; log.append(("1 глагол", s))
            else:
                w, s = strip(w, rv, [], NOUN)
                if s: log.append(("1 существительное", s))
    if w.endswith("и") and len(w) - 1 >= rv:
        w = w[:-1]; log.append(("2 и", "и"))
    w, s = strip(w, r2, [], DERIVATIONAL)
    if s: log.append(("3 словообразование", s))
    s = longest(w, rv, ["ейш", "ейше"])
    if s:
        w = w[: -len(s)]; log.append(("4 превосходная", s))
    if w.endswith("нн") and len(w) - 2 >= rv:
        w = w[:-1]; log.append(("4 нн → н", "н"))
    elif w.endswith("ь") and len(w) - 1 >= rv:
        w = w[:-1]; log.append(("4 мягкий знак", "ь"))
    return (w, log) if trace else w

for word in ["книгами", "красивейшими", "улыбнувшись", "радостный", "прочитавшего", "стали", "сталью", "люди"]:
    stem, log = snowball_ru(word, trace=True)
    rv, r2 = regions(word)
    steps = "; ".join(f"{name}: -{s}" for name, s in log) or "ничего не снято"
    print(f"{word:<14} RV={word[rv:]:<10} R2={word[r2:]:<8} → {stem:<10} [{steps}]")

Сверим с готовым стеммером `nltk.stem.SnowballStemmer("russian")` на всех формах нескольких слов. Формы возьмём из словаря `pymorphy3` (раздел 2): так проверка охватит сотни разных окончаний.

In [ ]:
!pip install -q pymorphy3

In [ ]:
import pymorphy3
from nltk.stem.snowball import SnowballStemmer

morph = pymorphy3.MorphAnalyzer()
nltk_stem = SnowballStemmer("russian")

lemmas = ["стол", "книга", "новый", "читать", "улыбаться", "прыгнуть", "радость", "написанный",
          "кричащий", "лучший", "человек", "здание", "армия", "синий", "бегущий", "прочитавший"]
forms = sorted({f.word for lemma in lemmas for f in morph.parse(lemma)[0].lexeme})
mismatch = [w for w in forms if snowball_ru(w) != nltk_stem.stem(w)]
print(f"форм проверено: {len(forms)}, расхождений с nltk: {len(mismatch)}")
print("пример:", [(w, nltk_stem.stem(w)) for w in forms[::90]])

## 2. Морфологический анализ: `pymorphy3`

**Морфологический анализатор** по словоформе возвращает лемму (начальную форму) и грамматические признаки (тег). `pymorphy3` использует словарь OpenCorpora: для каждой формы он знает все возможные разборы, а `score` — оценка вероятности разбора, полученная по размеченному корпусу. Для слов, которых нет в словаре, он угадывает разбор по концу слова.

Неоднозначность здесь та же, что в лаборатории: «стали» — глагол «стать» или существительное «сталь», «мыла» — «мыть» или «мыло».

In [ ]:
for word in ["стали", "мыла", "душ", "печь", "люди"]:
    print(word)
    for p in morph.parse(word)[:4]:
        print(f"   {p.normal_form:<8} {p.tag.cyr_repr:<34} score={p.score:.3f}")

Тег — это набор граммем: часть речи (`СУЩ`, `ГЛ`, `ИНФ`), род, число, падеж, время… Метод `inflect` ставит слово в нужную форму. Построим парадигму 6 падежей × 2 числа для слова «человек»: во множественном числе основа меняется целиком («люди»), это **супплетивизм**. Стеммер такое не свяжет никогда, а словарь знает. Во второй таблице («сталь») форма «стали» занимает пять клеток из двенадцати: отсюда и неоднозначность разбора.

In [ ]:
import pandas as pd

CASES = [("nomn", "им."), ("gent", "род."), ("datv", "дат."), ("accs", "вин."), ("ablt", "твор."), ("loct", "предл.")]

def paradigm(word):
    p = morph.parse(word)[0]
    rows = {}
    for case, name in CASES:
        rows[name] = [p.inflect({case, num}).word for num in ("sing", "plur")]
    return pd.DataFrame.from_dict(rows, orient="index", columns=["ед. ч.", "мн. ч."])

print(paradigm("человек"))
print()
print(paradigm("сталь"))

## 3. Стемминг или лемматизация?

Возьмём короткий текст и для каждого слова найдём стем (Snowball) и лемму (самый вероятный разбор `pymorphy3`). Сгруппируем слова двумя способами:

- **недорезание** (under-stemming): у слов одна лемма, но разные стемы («люди» и «человек»);
- **перерезание** (over-stemming): у слов один стем, но разные леммы («стали» и «сталь»).

In [ ]:
import re

text = """
Люди давно научились делать сталь. Из стали делают мосты, ножи и рельсы. Новые мосты стали прочнее и легче.
Человек шёл по мосту и смотрел на реку. Дети шли за ним и смеялись. Ребёнок нёс в руках мыло и полотенце.
Мама мыла раму, а папа мыл окна. Скоро окна стали чистыми. В печи пекли хлеб, и дом наполнился теплом.
Бабушка пекла пироги по выходным. Хороший хлеб лучше любого пирога. Лучшие пироги пекут в деревне.
Мы идём домой и несём горячий хлеб. Зимой люди любят тёплые печи. Человеку нужен дом, а детям нужна школа.
Школьники читают книги о людях и о стали. Хорошие книги учат детей думать.
"""
words = re.findall(r"[а-яё]+", text.lower())

In [ ]:
from collections import defaultdict

types = sorted(set(words))
stem_of = {w: snowball_ru(w) for w in types}
lemma_of = {w: morph.parse(w)[0].normal_form for w in types}
print(f"токенов {len(words)}, типов {len(types)}, стемов {len(set(stem_of.values()))}, лемм {len(set(lemma_of.values()))}")

by_lemma, by_stem = defaultdict(set), defaultdict(set)
for w in types:
    by_lemma[lemma_of[w]].add(w)
    by_stem[stem_of[w]].add(w)

print("\nНедорезание: одна лемма, разные стемы")
for lemma, ws in sorted(by_lemma.items()):
    stems = sorted({stem_of[w] for w in ws})
    if len(stems) > 1:
        print(f"   {lemma:<9} формы: {', '.join(sorted(ws)):<32} стемы: {', '.join(stems)}")
print("\nПеререзание: один стем, разные леммы")
for stem, ws in sorted(by_stem.items()):
    lemmas = sorted({lemma_of[w] for w in ws})
    if len(lemmas) > 1:
        print(f"   {stem:<6} формы: {', '.join(sorted(ws)):<20} леммы: {', '.join(lemmas)}")

Лемматизатор тоже ошибается: он смотрит на слово без контекста и берёт самый частый разбор. В тексте «Мама **мыла** раму» и «папа **мыл** окна» — это глагол «мыть», а «бабушка **пекла**» — глагол «печь», но выбрано существительное. В «Из **стали** делают мосты» наоборот: выбран глагол «стать», хотя здесь существительное «сталь». Найдём такие места: слова, у которых среди разборов есть глагол, а выбран не глагол. Снять эту неоднозначность помогает контекст (разметка частей речи, тема 8).

In [ ]:
import matplotlib.pyplot as plt

for w in ["мыла", "мыл", "пекла", "стали"]:
    parses = morph.parse(w)
    verbs = [p for p in parses if p.tag.POS in ("VERB", "INFN")]
    best = parses[0]
    flag = "  ← выбран не глагол" if verbs and best.tag.POS not in ("VERB", "INFN") else ""
    print(f"{w:<6} лучший разбор: {best.normal_form} ({best.tag.cyr_repr.split(',')[0]}, {best.score:.2f}); "
          f"глагольный: {verbs[0].normal_form if verbs else '—'} ({verbs[0].score:.2f}){flag}" if verbs else w)

counts = [len(types), len(set(stem_of.values())), len(set(lemma_of.values()))]
plt.figure(figsize=(5, 3))
plt.bar(["типы", "стемы Snowball", "леммы pymorphy3"], counts, color=["#888888", "#2B55C9", "#17907E"])
for i, c in enumerate(counts):
    plt.text(i, c + 1, str(c), ha="center")
plt.title("Сколько разных единиц в тексте")
plt.ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.show()

## 4. Конечный автомат: основа → суффикс → окончание

Словарные анализаторы внутри устроены как **конечные автоматы** (FSA): состояния — места в слове, переходы — морфемы. Соберём маленький автомат для существительных мужского рода с твёрдой основой (стол, мост, стакан, кот, слон): основа, затем необязательный уменьшительный суффикс (-ик или -чик), затем **одно** окончание. Русский — фузионный язык: одно окончание выражает сразу падеж и число, поэтому автомат часто даёт несколько разборов. Орфографический алломорф: после к, г, х пишется «и», а не «ы» (столы, но столики). У одушевлённых существительных винительный падеж совпадает с родительным (вижу кота, но вижу стол).

In [ ]:
LEXICON = {"стол": ("ик", False), "мост": ("ик", False), "стакан": ("чик", False),
           "кот": ("ик", True), "слон": ("ик", True)}          # основа: (суффикс, одушевлённость)
ENDINGS = {  # (падеж, число): окончание
    ("им.", "ед."): "",    ("род.", "ед."): "а",  ("дат.", "ед."): "у",
    ("вин.", "ед."): None, ("твор.", "ед."): "ом", ("предл.", "ед."): "е",
    ("им.", "мн."): "ы",   ("род.", "мн."): "ов", ("дат.", "мн."): "ам",
    ("вин.", "мн."): None, ("твор.", "мн."): "ами", ("предл.", "мн."): "ах",
}

def ending(case, num, animate):
    if case == "вин.":   # вин. = род. у одушевлённых, = им. у неодушевлённых
        case = "род." if animate else "им."
    return ENDINGS[case, num]

def spell(stem, end):
    # после к, г, х пишется и, а не ы
    if end == "ы" and stem[-1] in "кгх":
        return "и"
    return end

def generate(root, diminutive=False):
    suf, animate = LEXICON[root]
    stem = root + (suf if diminutive else "")
    return {key: stem + spell(stem, ending(*key, animate)) for key in ENDINGS}

def analyze_ru(word):
    # проходим автомат: КОРЕНЬ → (СУФФИКС) → ОКОНЧАНИЕ; возвращаем все пути
    out = []
    for root, (suf, animate) in LEXICON.items():
        for dim in (False, True):
            stem = root + (suf if dim else "")
            if not word.startswith(stem):
                continue
            rest = word[len(stem):]
            for (case, num) in ENDINGS:
                if spell(stem, ending(case, num, animate)) == rest:
                    parts = [root] + ([suf] if dim else []) + [rest or "Ø"]
                    out.append((" + ".join(parts), case, num))
    return out

for w in ["столы", "кота", "стаканчиками", "мостике", "слоники", "стол", "столамик", "столикы"]:
    res = analyze_ru(w)
    print(w, "→", "; ".join(f"{p} [{c} {n}]" for p, c, n in res) if res else "отвергнуто автоматом")

Проверим автомат по словарю: сгенерируем все 12 форм каждого слова (и его уменьшительной формы) и сравним с `pymorphy3.inflect`. Затем нарисуем сам автомат: рамка — состояние, внутри — морфемы, по которым в него входят.

In [ ]:
CASE_TAG = {"им.": "nomn", "род.": "gent", "дат.": "datv", "вин.": "accs", "твор.": "ablt", "предл.": "loct"}
NUM_TAG = {"ед.": "sing", "мн.": "plur"}
total = errors = 0
for root, (suf, _) in LEXICON.items():
    for dim in (False, True):
        lemma = root + (suf if dim else "")
        p = morph.parse(lemma)[0]
        for (case, num), form in generate(root, dim).items():
            gold = p.inflect({CASE_TAG[case], NUM_TAG[num]}).word
            total += 1
            if gold != form:
                errors += 1
                print("расхождение:", lemma, case, num, form, gold)
print(f"форм сгенерировано: {total}, расхождений с pymorphy3: {errors}")
forms = generate("кот", True)
print(pd.DataFrame([[forms[c, n] for n in ("ед.", "мн.")] for c in CASE_TAG], index=list(CASE_TAG), columns=["ед. ч.", "мн. ч."]))

import textwrap

def draw_fsa(edges, pos, final, title):
    # рамка — состояние, внутри — морфемы, по которым в него входят
    incoming = {}
    for (a, b), sufs in edges.items():
        incoming.setdefault(b, set()).update(s or "Ø" for s in sufs)
    fig, ax = plt.subplots(figsize=(11, 4.6))
    boxes = {}
    for name, (x, y) in pos.items():
        label = name
        if name in incoming:
            label += "\n" + textwrap.fill(", ".join(sorted(incoming[name], key=lambda s: (len(s), s))), 22)
        boxes[name] = ax.text(x, y, label, ha="center", va="center", fontsize=8, zorder=3,
                              bbox=dict(boxstyle="round,pad=0.5", fc="#DCE6FA", ec="#2B55C9",
                                        lw=2.5 if name in final else 1))
    fig.canvas.draw()
    for (a, b) in edges:
        (x1, y1), (x2, y2) = pos[a], pos[b]
        rad = (-1 if y1 > 1 else 1) / abs(x2 - x1) if y1 == y2 and abs(x2 - x1) > 2.5 else 0.0
        ax.annotate("", xy=(0.5, 0.5), xycoords=boxes[b], xytext=(0.5, 0.5), textcoords=boxes[a],
                    arrowprops=dict(arrowstyle="-|>", color="#555555", lw=1.1, mutation_scale=14,
                                    patchA=boxes[a].get_bbox_patch(), patchB=boxes[b].get_bbox_patch(),
                                    connectionstyle=f"arc3,rad={rad}"))
    xs, ys = [p[0] for p in pos.values()], [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 0.9, max(xs) + 0.9)
    ax.set_ylim(min(ys) - 1.2, max(ys) + 1.2)
    ax.axis("off")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
RU_EDGES = {("КОРЕНЬ", "СУФФИКС"): ["ик", "чик"], ("КОРЕНЬ", "ОКОНЧАНИЕ"): [""], ("СУФФИКС", "ОКОНЧАНИЕ"): [""],
            ("ОКОНЧАНИЕ", "КОНЕЦ"): sorted({e for e in ENDINGS.values() if e is not None} | {"и"})}
draw_fsa(RU_EDGES, {"КОРЕНЬ": (0, 0), "СУФФИКС": (2, 1.2), "ОКОНЧАНИЕ": (4, 0), "КОНЕЦ": (6, 0)},
         final={"КОНЕЦ"}, title="Автомат для существительных: стол-ик-ами (жирная рамка — конечное состояние)")

## 5. Задания

1. Прогоните стеммер раздела 1 с `trace=True` по словам «бегущими», «учительница», «рассказывавшая», «новейшее». Для каждого слова объясните, почему сработал именно этот шаг и почему остаток не сократился ещё.
2. Добавьте в автомат раздела 4 существительные женского рода на -а (`лампа`, `книга`): им. -а, род. -ы/-и, дат. -е, вин. -у, твор. -ой, предл. -е; мн. -ы/-и, -Ø, -ам, -ы/-и, -ами, -ах. Проверьте по `pymorphy3`, как в разделе 4. Сколько разборов теперь у формы «книге»? А у «книги»?
3. Для всех слов текста раздела 3 посчитайте, сколько разных лемм предлагает `morph.parse` (без повторов). Какая доля слов неоднозначна? Выпишите пять самых неоднозначных.
4. Напишите `lemmatize_in_context(words)`: если перед словом стоит существительное или местоимение в им. п., а у слова есть глагольный разбор, выбирайте его. Сколько ошибок из раздела 3 («мыла», «мыл», «пекла») исправилось?
5. Постройте с помощью `inflect` парадигму глагола «идти» в настоящем времени (3 лица × 2 числа) и прошедшем (м., ж., ср. род и мн. ч.). Какие формы стеммер Snowball не сведёт к общему стему?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь